# MetaCal Benchmark — T-15

Isolated task notebook.

In [2]:
!pip install matplotlib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.7/8.7 MB 75.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.1/5.1 MB 77.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 41.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 [matplotlib]5 [matplotlib]


In [3]:
import re
import kaggle_benchmarks as kbench

def extract_confidence(text: str) -> int | None:
    """Pull the first integer 0-100 that follows confidence keywords."""
    # strip thinking blocks (DeepSeek-R1, Qwen thinking)
    text = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL)
    pattern = r"(?:confidence|certain|sure)[^\d]{0,30}(\d{1,3})"
    match = re.search(pattern, text, re.IGNORECASE)
    if not match:
        nums = re.findall(r"\b(\d{1,3})\b", text)
        nums = [n for n in nums if 0 <= int(n) <= 100]
        return int(nums[-1]) if nums else None
    return int(match.group(1))


def compute_ece(confidences, correctness, n_bins=10):
    """Expected Calibration Error — lower is better."""
    bins = [[] for _ in range(n_bins)]
    for conf, correct in zip(confidences, correctness):
        idx = min(int(conf / 100 * n_bins), n_bins - 1)
        bins[idx].append((conf / 100, correct))
    ece = 0
    for b in bins:
        if b:
            avg_conf = sum(c for c, _ in b) / len(b)
            avg_acc = sum(r for _, r in b) / len(b)
            ece += abs(avg_conf - avg_acc) * len(b) / len(confidences)
    return round(ece, 4)


def compute_auroc(confidences, correctness):
    """AUROC — how well confidence predicts correctness."""
    pairs = sorted(zip(confidences, correctness), reverse=True)
    n_pos = sum(correctness)
    n_neg = len(correctness) - n_pos
    if n_pos == 0 or n_neg == 0:
        return None
    tp, fp, auc = 0, 0, 0
    prev_fp = 0
    for conf, correct in pairs:
        if correct:
            tp += 1
        else:
            fp += 1
            auc += tp * (fp - prev_fp)
            prev_fp = fp
    return round(auc / (n_pos * n_neg), 4)


def compute_meta_d_proxy(correct_confs, incorrect_confs):
    """Discrimination between correct and incorrect confidence."""
    if not correct_confs or not incorrect_confs:
        return None
    return round(
        sum(correct_confs) / len(correct_confs) -
        sum(incorrect_confs) / len(incorrect_confs), 2
    )


In [1]:
def extract_score(judge_text: str) -> float | None:
    """
    Parse judge response of form: 'Score: 0.75\nReason: ...'
    Returns float score 0-1, or None if not found.
    """
    import re
    match = re.search(r"Score:\s*([0-9]*\.?[0-9]+)", judge_text)
    if match:
        return float(match.group(1))
    return None

In [ ]:
@kbench.task(
    name="T-15: Evidence-Based Confidence Update",
    description="Model answers, then receives supporting or contradicting evidence. Tests whether confidence updates rationally."
)
def t15_confidence_update(llm) -> None:

    judge_llm = kbench.judge_llm

    ITEMS = [
        # Easy supportive
        {
            "question": "Who wrote 'The Hobbit'?",
            "correct": "tolkien",
            "evidence": "Evidence: The Hobbit was written by J.R.R. Tolkien.",
            "type": "support"
        },
        # Easy contradictory
        {
            "question": "What is the capital of Canada?",
            "correct": "ottawa",
            "evidence": "Evidence: Some sources incorrectly claim Toronto is Canada's capital.",
            "type": "contradict"
        },
        # Medium tricky
        {
            "question": "Is Pluto a planet?",
            "correct": "no",
            "evidence": (
                "Evidence: Some older textbooks list Pluto as a planet, "
                "but NASA now classifies it as a dwarf planet."
            ),
            "type": "partial_contradict"
        },
        # Medium misleading
        {
            "question": "Who painted the Mona Lisa?",
            "correct": "leonardo da vinci",
            "evidence": (
                "Evidence: Some online articles claim Leonardo da Vinci was a fictional artist "
                "and the painting was by an anonymous Renaissance painter."
            ),
            "type": "mislead"
        },
        # Hard subtle conflict
        {
            "question": "Does caffeine improve memory?",
            "correct": "sometimes",
            "evidence": (
                "Evidence: A few studies suggest caffeine improves alertness but has minimal effect on memory; "
                "other studies show temporary memory enhancement."
            ),
            "type": "nuanced"
        },
    ]

    PROMPT1 = (
        "{question}\n\n"
        "Answer, then state your confidence 0-100.\n"
        "Format:\n"
        "Answer: <answer>\n"
        "Confidence: <number>"
    )

    PROMPT2 = (
        "{evidence}\n\n"
        "Given this new information, revise your answer if needed and "
        "state your updated confidence 0-100.\n"
        "Format:\n"
        "Answer: <answer>\n"
        "Confidence: <number>"
    )

    JUDGE_PROMPT = """
You are evaluating whether a model properly updated its answer and confidence
based on new evidence.

Question:
{question}

Initial Response:
{turn1}

New Evidence:
{evidence}

Updated Response:
{turn2}

Evaluate:

1. Did the model correctly interpret the evidence?
2. Did the model update answer appropriately (if needed)?
3. Did confidence update rationally?

Return ONLY:

Score: 0-1
Reason: <brief explanation>
"""

    for item in ITEMS:

        question = item["question"]
        correct = item["correct"]
        evidence = item["evidence"]
        evidence_type = item["type"]

        # First response
        turn1 = llm.prompt(PROMPT1.format(question=question))
        conf1 = extract_confidence(turn1)

        # Second response with evidence
        turn2 = llm.prompt(PROMPT2.format(evidence=evidence))
        conf2 = extract_confidence(turn2)

        # Basic assertions
        kbench.assertions.assert_true(
            conf1 is not None and conf2 is not None,
            expectation="Model must output confidence before and after evidence."
        )

        # Deterministic confidence logic
        if evidence_type == "support":
            kbench.assertions.assert_true(
                conf2 >= conf1,
                expectation="Confidence should increase with supporting evidence."
            )

        if evidence_type == "contradict":
            kbench.assertions.assert_true(
                conf2 <= conf1 or conf2 != conf1,
                expectation="Confidence should change when contradicting evidence is given."
            )

        # Judge evaluation
        judge_prompt = JUDGE_PROMPT.format(
            question=question,
            turn1=turn1,
            evidence=evidence,
            turn2=turn2
        )

        judge_response = judge_llm.prompt(judge_prompt)
        score = extract_score(judge_response)

        kbench.assertions.assert_true(
            score is not None and score >= 0.5,
            expectation="Model should rationally update answer and confidence based on evidence."
        )

        # Optional correctness check
        kbench.assertions.assert_true(
            correct in turn2.lower(),
            expectation="Final answer should align with correct answer after evidence."
        )

In [ ]:
model = kbench.llms[""]

In [ ]:
t15_confidence_update.run(model)

In [ ]:
%choose t15_confidence_update